# Study 04 ENNI Context Ablation

This notebook runs the pilot `prev_same_speaker` ENNI context-window experiment from this repo and compares it against the existing `utterance_only` outputs.

Workflow:
1. Clone your repo branch.
2. Install inference dependencies.
3. Set your Hugging Face token.
4. Build the pilot ENNI evaluation JSONL from the clean `.cha` files and the 10-file manifest.
5. Reuse the existing `utterance_only` predictions.
6. Run the model once in `prev_same_speaker` mode.
7. Compare the outputs on shared `row_id`s.


In [ ]:
REPO_URL = "https://github.com/shamira-venturini/talkbank-morphosyntax-error-annotator.git"
BRANCH = "<YOUR-BRANCH>"
REPO_DIR = "/content/talkbank-morphosyntax-error-annotator"

# Set LIMIT = 50 for a smoke test, then switch to 0 for the full pilot.
LIMIT = 50
BATCH_SIZE = 4

PILOT_FILE_MANIFEST = "study_04_context_windows/data/pilot_10_ambiguous_agreement_over_past_files.txt"
CLEAN_ENNI_DIR = "studies/04_context_windows/ENNI"
PILOT_INPUT_JSONL = "study_04_context_windows/data/context_eval/pilot_10_prev_same_speaker_eval.jsonl"
PILOT_SUMMARY_JSON = "study_04_context_windows/data/context_eval/pilot_10_prev_same_speaker_eval_summary.json"
STAGE3_SPLIT = "study_01_talkbank_tool_paper/experiments/recon_full_comp_preserve/stage3_train.jsonl"
CHAT_TOKENS = "study_01_talkbank_tool_paper/experiments/recon_full_comp_preserve/chat_tokens.json"
PREV_OUTPUT_DIR = "study_04_context_windows/results/enni_context_ablation_pilot_10_prev_same_speaker"

# Existing utterance_only baseline. The analysis script compares on shared row_ids.
EXISTING_UTTERANCE_ONLY_JSONL = "study_02_hitl_adaptation/results/OOD_eval/enni_ood_chat/predictions_utterance_only.jsonl"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content
!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!git checkout "$BRANCH"
!git status --short


In [ ]:
!nvidia-smi
!python3 --version


In [ ]:
!pip install -q transformers peft bitsandbytes accelerate sentencepiece scipy orjson


In [ ]:
import os
import getpass

os.environ["HF_TOKEN"] = getpass.getpass("HF token: ")


In [ ]:
%cd "$REPO_DIR"
!python3 study_04_context_windows/scripts/prepare_enni_context_ablation_input.py \
  --enni-dir "$CLEAN_ENNI_DIR" \
  --file-manifest "$PILOT_FILE_MANIFEST" \
  --out-jsonl "$PILOT_INPUT_JSONL" \
  --out-summary "$PILOT_SUMMARY_JSON" \
  --limit "$LIMIT"


Validate the existing `utterance_only` prediction file. This notebook does not rerun that condition.


In [ ]:
import json
from pathlib import Path

utterance_only_path = Path(REPO_DIR) / EXISTING_UTTERANCE_ONLY_JSONL
pilot_summary_path = Path(REPO_DIR) / PILOT_SUMMARY_JSON

print("utterance_only baseline:", utterance_only_path)
print("exists:", utterance_only_path.exists())
print("pilot summary:", pilot_summary_path)
print(pilot_summary_path.read_text(encoding="utf-8"))


Run the frozen model once on the pilot JSONL in `prev_same_speaker` mode.


In [ ]:
%cd "$REPO_DIR"
!python3 study_04_context_windows/scripts/run_ood_context_inference.py \
  --input-jsonl "$PILOT_INPUT_JSONL" \
  --context-mode prev_same_speaker \
  --stage3-split "$STAGE3_SPLIT" \
  --chat-tokens "$CHAT_TOKENS" \
  --out-dir "$PREV_OUTPUT_DIR" \
  --batch-size "$BATCH_SIZE" \
  --limit "$LIMIT"


Compare the existing `utterance_only` predictions against the new `prev_same_speaker` predictions. The main summary is written to `analysis/summary.json`.


In [ ]:
%cd "$REPO_DIR"
!python3 study_04_context_windows/scripts/analyze_enni_context_ablation.py \
  --utterance-only "$EXISTING_UTTERANCE_ONLY_JSONL" \
  --prev-same-speaker "$PREV_OUTPUT_DIR/predictions_prev_same_speaker.jsonl" \
  --out-dir "$PREV_OUTPUT_DIR/analysis"


In [ ]:
import json
from pathlib import Path

summary_path = Path(REPO_DIR) / PREV_OUTPUT_DIR / "analysis" / "summary.json"
print(summary_path)
print(summary_path.read_text(encoding="utf-8"))


For the full pilot, set `LIMIT = 0` in the config cell and rerun the prep cell, the `prev_same_speaker` inference cell, and the analysis cell. Do not rerun `utterance_only` unless you intentionally want to replace the baseline file.
